# Supplementary Tables 22–24: regenerated from current data, with Source Data

**Input:** `260926_perdomain_annotations_2.csv` (one row per GH18 domain; `query_name` = InDel-class HMM hit).

**Choices, set in Cell 1. State them in the legends.**
* **Unit (Tables 22–23):** one count per *sequence per InDel class*. A protein whose GH18 domains fall
  in two classes counts once in each. GH18 domains with no class hit are excluded.
* **Missing phylum** is reported as `Unassigned`. It is shown in Table 22 but excluded from the
  Table 23 tests, since it isn't a phylum.
* **Table 24:** auxiliary-domain annotations of each sequence in the class, one count per annotation,
  in the category scheme of the input's `broad_categories` column (currently the 12-category Global scheme).
* **Tests:** class vs all other classes, 2×2 table, log2 OR with +0.5 continuity correction, two-sided
  Fisher's exact test, Benjamini–Hochberg FDR over all tests in a table.
* **Rows displayed in Tables 23–24:** FDR < 0.05, up to 5 enriched and 5 depleted per class, ranked by log2 OR.
  Every test is kept in the Source Data.

for UNIT variable: 
"sequence_x_class" (default), one protein in one class	counts 1 in C	counts 1 in B and 1 in C	39,827
"domain"	one GH18 domain	counts 2 in C	counts 1 in B and 1 in C	39,946

T24_COUNT

Setting	That protein with 3 CBMs adds to "CBMs"	Measures
"annotation" (default)	3	the composition of all auxiliary annotations (the published Table 24 design: "proportional composition")
"carriage"	1	how many sequences carry at least one domain of that category
Which to choose: "annotation" answers "what fraction of the class's auxiliary domains are CBMs?". "carriage" answers "what fraction of the class's sequences carry a CBM?".

In [ ]:
# Cell 1 — configuration
import os, numpy as np, pandas as pd
from scipy.stats import fisher_exact

INFILE    = "../data/annotation/260926_perdomain_annotations_2.csv"
OUT_DIR   = "../results/source_data/source_data_STable22_24"
CID2CLASS = {"CID2": "A", "CID1and0": "B", "CID3": "C", "CID4": "D", "CID5": "E", "CID6": "F", "CID7": "G"}
UNIT      = "sequence_x_class"        # or "domain" (one count per GH18 domain)
T24_COUNT = "annotation"              # or "carriage" (category counted once per sequence)
FDR_MAX, N_SHOW = 0.05, 5
os.makedirs(OUT_DIR, exist_ok=True)

d = pd.read_csv(INFILE, low_memory=False,
                usecols=["Entry", "fastaID", "query_name", "phylum", "broad_categories"])
d["indel_class"] = d.query_name.map(CID2CLASS)
d["phylum"] = d.phylum.fillna("Unassigned")
print(f"{len(d):,} GH18 domains | {d.Entry.nunique():,} sequences | "
      f"{d.indel_class.isna().sum():,} domains without an InDel-class hit (excluded)")

base = d.dropna(subset=["indel_class"])
units = base.drop_duplicates(["Entry", "indel_class"]) if UNIT == "sequence_x_class" else base
print(f"counting units: {len(units):,}"); display(units.indel_class.value_counts().sort_index().to_frame("n"))

In [ ]:
# Cell 2 — shared functions
def bh(p):
    p = np.asarray(p, float); n = len(p); o = np.argsort(p); q = np.empty(n)
    q[o] = np.minimum.accumulate((p[o] * n / np.arange(1, n + 1))[::-1])[::-1]
    return np.minimum(q, 1)

def enrichment(tab):
    N, tot = tab.sum(), tab.values.sum(); rows = []
    for c in tab.columns:
        for k in tab.index:
            a = int(tab.loc[k, c]); b = N[c] - a; cc = int(tab.loc[k].sum()) - a; dd = tot - N[c] - cc
            rows.append({"InDel_class": c, "level": k, "count": a, "class_total": int(N[c]),
                         "pct_of_class": 100 * a / N[c],
                         "log2_OR": np.log2(((a + .5) * (dd + .5)) / ((b + .5) * (cc + .5))),
                         "p": fisher_exact([[a, b], [cc, dd]])[1]})
    r = pd.DataFrame(rows); r["FDR"] = bh(r.p)
    r["direction"] = np.where(r.log2_OR > 0, "enriched", "depleted")
    return r

def select(r):
    sig = r[r.FDR < FDR_MAX]
    out = [pd.concat([g[g.direction == "enriched"].nlargest(N_SHOW, "log2_OR"),
                      g[g.direction == "depleted"].nsmallest(N_SHOW, "log2_OR")]) for _, g in sig.groupby("InDel_class")]
    return pd.concat(out).reset_index(drop=True)

def fmt(r, level_name):
    return pd.DataFrame({"InDel Class": r.InDel_class, level_name: r.level, "Direction": r.direction,
                         "Count (% of class)": [f"{n} ({p:.2g}%)" for n, p in zip(r["count"], r.pct_of_class)],
                         "Log2(odds ratio)": r.log2_OR.round(2),
                         "p-value": r.p.map(lambda v: f"{v:.2g}"), "FDR (p-value)": r.FDR.map(lambda v: f"{v:.2g}")})

In [ ]:
# Cell 3 — Supplementary Table 22 (top 5 phyla per class)
tab_ph = pd.crosstab(units.phylum, units.indel_class)
t22 = []
for c in tab_ph.columns:
    s = tab_ph[c][tab_ph[c] > 0].sort_values(ascending=False).head(5)
    t22 += [{"InDel Class": c, "Phylum": k, "count": int(v),
             "Proportion of InDel Class (%)": round(100 * v / tab_ph[c].sum(), 2)} for k, v in s.items()]
t22 = pd.DataFrame(t22); display(t22)

In [ ]:
# Cell 4 — Supplementary Table 23 (phylum enrichment; Unassigned excluded from tests)
r23 = enrichment(tab_ph.drop(index="Unassigned", errors="ignore"))
t23 = fmt(select(r23), "Phylum"); display(t23)

In [ ]:
# Cell 5 — Supplementary Table 24 (auxiliary-domain category enrichment)
seqs = units[["Entry", "indel_class", "broad_categories"]].copy()
ann = (seqs.assign(broad_category=seqs.broad_categories.str.split("|"))
           .explode("broad_category").dropna(subset=["broad_category"]).reset_index(drop=True))
if T24_COUNT == "carriage":
    ann = ann.drop_duplicates(["Entry", "indel_class", "broad_category"])
tab_cat = pd.crosstab(ann.broad_category, ann.indel_class)
r24 = enrichment(tab_cat)
t24 = fmt(select(r24), "Broad category").drop(columns=["p-value"]); display(t24)

In [ ]:
# Cell 6 — Source Data (per-unit data + all tests), all traceable to INFILE
units[["Entry", "fastaID", "query_name", "indel_class", "phylum"]].to_csv(
    f"{OUT_DIR}/SourceData_STable22_23_per_sequence.csv", index=False, encoding="utf-8-sig")
r23.to_csv(f"{OUT_DIR}/SourceData_STable23_all_tests.csv", index=False, encoding="utf-8-sig")
ann[["Entry", "indel_class", "broad_category"]].to_csv(
    f"{OUT_DIR}/SourceData_STable24_per_annotation.csv", index=False, encoding="utf-8-sig")
r24.to_csv(f"{OUT_DIR}/SourceData_STable24_all_tests.csv", index=False, encoding="utf-8-sig")
for name, t in [("STable22", t22), ("STable23", t23), ("STable24", t24)]:
    t.to_csv(f"{OUT_DIR}/{name}_regenerated.csv", index=False, encoding="utf-8-sig")

# round-trip: the displayed tables must be recomputable from the per-unit Source Data
chk = pd.read_csv(f"{OUT_DIR}/SourceData_STable22_23_per_sequence.csv", encoding="utf-8-sig")
assert pd.crosstab(chk.phylum, chk.indel_class).equals(tab_ph)
chk = pd.read_csv(f"{OUT_DIR}/SourceData_STable24_per_annotation.csv", encoding="utf-8-sig")
assert pd.crosstab(chk.broad_category, chk.indel_class).equals(tab_cat)
print("written:", sorted(os.listdir(OUT_DIR)))